In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 🧪 MLOps API Automated Testing Notebook\n",
    "\n",
    "Notebook ini digunakan untuk melakukan pengujian otomatis (*integration test*) terhadap sistem inferensi Machine Learning **Credit Card Fraud Detection** yang telah di-deploy secara live di **Railway**.\n",
    "\n",
    "**Target API Domain:** `https://mlops-creditcardfraud-detection-production.up.railway.app`"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Impor Dependensi & Konfigurasi Base URL"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "metadata": {},
   "outputs": [],
   "source": [
    "import requests\n",
    "import json\n",
    "import random\n",
    "import pandas as pd\n",
    "\n",
    "# Endpoint URL Publik di Cloud Railway\n",
    "BASE_URL = \"https://mlops-creditcardfraud-detection-production.up.railway.app\"\n",
    "print(f\"Target API Base URL: {BASE_URL}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Pengujian Endpoint Health Check (`GET /health`)\n",
    "Memastikan API aktif dan model TensorFlow Servable telah berhasil dimuat ke dalam memori server."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 2,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Mengirimkan request GET ke endpoint health\n",
    "response_health = requests.get(f\"{BASE_URL}/health\")\n",
    "\n",
    "print(\"Status Code:\", response_health.status_code)\n",
    "print(\"Response Body:\", json.dumps(response_health.json(), indent=2))\n",
    "\n",
    "# Verifikasi Assertions\n",
    "assert response_health.status_code == 200, f\"Expected 200 OK, got {response_health.status_code}\"\n",
    "assert response_health.json().get(\"status\") in [\"healthy\", \"ok\"], \"Service status is not healthy!\"\n",
    "print(\"\n Health Check Test Passed Successfully!\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Pengujian Endpoint Prediksi dengan Samples Transaksi (`POST /predict`)\n",
    "Mengirimkan payload JSON fitur transaksi ke API dan menerima hasil inferensi berupa probabilitas serta prediksinya."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 3,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Payload Sampel Transaksi Normal\n",
    "sample_transaction = {\n",
    "    \"Time\": 0,\n",
    "    \"V1\": -1.359807, \"V2\": -0.072781, \"V3\": 2.536347, \"V4\": 1.378155,\n",
    "    \"V5\": -0.338321, \"V6\": 0.462388, \"V7\": 0.239599, \"V8\": 0.098698,\n",
    "    \"V9\": 0.363787, \"V10\": 0.090794, \"V11\": -0.551600, \"V12\": -0.617801,\n",
    "    \"V13\": -0.991390, \"V14\": -0.311169, \"V15\": 1.468177, \"V16\": -0.470401,\n",
    "    \"V17\": 0.207971, \"V18\": 0.025791, \"V19\": 0.403993, \"V20\": 0.251412,\n",
    "    \"V21\": -0.018307, \"V22\": 0.277838, \"V23\": -0.110474, \"V24\": 0.066928,\n",
    "    \"V25\": 0.128539, \"V26\": -0.189115, \"V27\": 0.133558, \"V28\": -0.021053,\n",
    "    \"Amount\": 149.62\n",
    "}\n",
    "\n",
    "# Mengirimkan HTTP POST Request\n",
    "headers = {\"Content-Type\": \"application/json\"}\n",
    "response_predict = requests.post(f\"{BASE_URL}/predict\", json=sample_transaction, headers=headers)\n",
    "\n",
    "print(\"HTTP Status Code:\", response_predict.status_code)\n",
    "print(\"Prediction Output:\", json.dumps(response_predict.json(), indent=2))\n",
    "\n",
    "# Verifikasi Assertions\n",
    "assert response_predict.status_code == 200, f\"Prediction request failed with status {response_predict.status_code}\"\n",
    "assert \"is_fraud\" in response_predict.json() or \"prediction\" in response_predict.json(), \"Prediction key missing in response!\"\n",
    "print(\"\n Single Prediction Test Passed Successfully!\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Pengujian Simulasi Pengiriman Data Acak (Dynamic Request Test)\n",
    "Pengujian stabilitas API dengan membuat variasi fitur acak."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 4,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Membuat payload transaksi acak sederhana\n",
    "dynamic_payload = {\n",
    "    \"Time\": random.randint(0, 172792),\n",
    "    **{f\"V{i}\": round(random.uniform(-3.0, 3.0), 6) for i in range(1, 29)},\n",
    "    \"Amount\": round(random.uniform(1.0, 5000.0), 2)\n",
    "}\n",
    "\n",
    "print(\"Sending Dynamic Payload Sample:\")\n",
    "print(json.dumps(dynamic_payload, indent=2))\n",
    "\n",
    "response_dynamic = requests.post(f\"{BASE_URL}/predict\", json=dynamic_payload)\n",
    "\n",
    "print(\"\nHTTP Status Code:\", response_dynamic.status_code)\n",
    "print(\"Response JSON:\", response_dynamic.json())\n",
    "\n",
    "assert response_dynamic.status_code == 200, \"Dynamic testing failed!\"\n",
    "print(\"\n Dynamic Payload Test Passed Successfully!\")"
   ]
  }
 ],
 "metadata": {
  "language_info": {
   "name": "python"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 2
}